# 03 — Face Embedding Extraction

## Overview

This notebook evaluates three **pretrained face-recognition models** on the standardized baseline and compressed facial images produced by the previous stages.

The models used in the original experiment are:

- **ArcFace condition** — loaded through InsightFace using the `antelopev2` model pack
- **MobileFaceNet condition** — loaded through InsightFace using the `buffalo_sc` model pack
- **MagFace** — ResNet-100 backbone loaded from pretrained MagFace weights

> **Important:** No face-recognition model is trained or fine-tuned in this notebook. The models are used only to extract face embeddings from the experimental images.

### Pipeline

L0 baseline images + L1–L5 compressed probe images  
→ image-format decoding  
→ pretrained face-recognition model  
→ face embedding  
→ cosine-similarity pair generation  
→ `raw_pairwise_scores.csv`

Although this notebook is named **embedding extraction**, the original working implementation also performs the first pairwise cosine-similarity generation step immediately after extraction. That behavior is preserved here because the resulting `raw_pairwise_scores.csv` is the input to `04_score_generation_and_fairness.ipynb`.

> **Privacy:** Participant facial images remain private and are not included in the public repository. Stored notebook outputs are cleared before publication.

## Environment Setup

The original experiment was developed in Google Colab.

This stage requires:

- `insightface` and `onnxruntime` for the InsightFace recognition models
- PyTorch and TorchVision for MagFace inference
- OpenCV and NumPy for image processing
- `djxl` for decoding JPEG XL probe images
- `heif-convert` for decoding HEIC probe images
- pandas and scikit-learn for storing and calculating pairwise cosine similarities

JPEG XL **v0.11.1** is retained to remain consistent with the compression stage.

In [ ]:
print("--- Installing face-recognition and image-decoding dependencies ---")

!pip install -q insightface onnxruntime
!apt-get update
!apt-get install -y libheif-examples

print("✅ Core dependencies installed.")

## Library Imports

In [ ]:
import argparse
import importlib.util
import io
import os
import shutil
import subprocess
import sys
import tarfile
import zipfile

import cv2
import numpy as np
import pandas as pd
import requests
import torch

from insightface.app import FaceAnalysis
from sklearn.metrics.pairwise import cosine_similarity
from torchvision import transforms

## Project Paths

The public repository does not contain the participant images.

The notebook expects the following private/local project structure:

```text
data/
├── preprocessed_images/
└── compressed_images/

models/
└── magface_epoch_00025.pth

results/
└── data/
```

- `preprocessed_images/` contains the L0 baseline PNG images.
- `compressed_images/` contains JPEG, JPEG XL, and HEIC L1–L5 probes.
- `magface_epoch_00025.pth` contains the pretrained MagFace weights used by the original experiment.
- `results/data/` stores the derived raw pairwise score table.

The MagFace weights are required for reproducing the MagFace condition, but should be reviewed separately for file size and redistribution/licensing considerations before being added to GitHub.

In [ ]:
BASELINE_DIR = "../data/preprocessed_images"
PROBE_DIR = "../data/compressed_images"

MODELS_DIR = "../models"
RESULTS_DIR = "../results/data"

MAGFACE_WEIGHTS_FILENAME = "magface_epoch_00025.pth"
MAGFACE_WEIGHTS_PATH = os.path.join(
    MODELS_DIR,
    MAGFACE_WEIGHTS_FILENAME,
)

RAW_SCORES_PATH = os.path.join(
    RESULTS_DIR,
    "raw_pairwise_scores.csv",
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True,
)

print(f"Baseline directory: {BASELINE_DIR}")
print(f"Probe directory: {PROBE_DIR}")
print(f"Results directory: {RESULTS_DIR}")

## JPEG XL Decoder Setup

JPEG XL probe images cannot be read directly by the OpenCV workflow used in the original experiment.

The notebook therefore installs the `djxl` decoder from the same **JPEG XL v0.11.1** static release used during compression.

`djxl` is used only to decode `.jxl` files to a temporary PNG before embedding extraction.

In [ ]:
JXL_VERSION = "v0.11.1"
JXL_FILENAME = f"jxl-linux-x86_64-static-{JXL_VERSION}.tar.gz"

JXL_DOWNLOAD_URL = (
    f"https://github.com/libjxl/libjxl/releases/download/"
    f"{JXL_VERSION}/{JXL_FILENAME}"
)

JXL_INSTALL_DIR = "/content/jxl_install"
DJXL_PATH = "/content/djxl"


def install_djxl_decoder():
    """Install the JPEG XL decoder used by the original experiment."""

    print(f"Downloading JPEG XL tools ({JXL_VERSION})...")

    response = requests.get(
        JXL_DOWNLOAD_URL,
        stream=True,
        timeout=120,
    )
    response.raise_for_status()

    tar_path = os.path.join(
        "/content",
        JXL_FILENAME,
    )

    with open(tar_path, "wb") as file:
        for chunk in response.iter_content(
            chunk_size=8192
        ):
            file.write(chunk)

    print("Extracting JPEG XL tools...")

    os.makedirs(
        JXL_INSTALL_DIR,
        exist_ok=True,
    )

    with tarfile.open(
        tar_path,
        "r:gz",
    ) as archive:
        archive.extractall(
            JXL_INSTALL_DIR,
        )

    decoder_source = None

    for root, _, files in os.walk(
        JXL_INSTALL_DIR
    ):
        if "djxl" in files:
            decoder_source = os.path.join(
                root,
                "djxl",
            )
            break

    if decoder_source is None:
        raise FileNotFoundError(
            "Could not locate the djxl decoder "
            "inside the JPEG XL release archive."
        )

    if os.path.exists(DJXL_PATH):
        os.remove(DJXL_PATH)

    shutil.move(
        decoder_source,
        DJXL_PATH,
    )

    os.chmod(
        DJXL_PATH,
        0o755,
    )

    if os.path.exists(tar_path):
        os.remove(tar_path)

    if os.path.exists(JXL_INSTALL_DIR):
        shutil.rmtree(
            JXL_INSTALL_DIR
        )

    print(f"✅ JPEG XL decoder ready: {DJXL_PATH}")


install_djxl_decoder()

In [ ]:
print("--- Verifying JPEG XL decoder ---")

!{DJXL_PATH} --version

## MagFace Repository and Pretrained Weights

The MagFace condition uses the official MagFace repository together with the pretrained weight file used in the original experiment.

The repository is downloaded so that its model definition can be loaded locally.

The notebook does **not** train MagFace. It constructs the published network architecture and loads the pretrained weights.

In [ ]:
MAGFACE_REPOSITORY_URL = (
    "https://github.com/IrvingMeng/"
    "MagFace/archive/refs/heads/main.zip"
)

MAGFACE_REPOSITORY_DIR = "MagFace"


def download_magface_repository():
    """Download and prepare the official MagFace source repository."""

    for folder in [
        "MagFace",
        "MagFace-main",
        "MagFace-master",
    ]:
        if os.path.exists(folder):
            shutil.rmtree(folder)

    print(
        "Downloading MagFace from the official "
        "GitHub repository..."
    )

    response = requests.get(
        MAGFACE_REPOSITORY_URL,
        timeout=120,
    )
    response.raise_for_status()

    with zipfile.ZipFile(
        io.BytesIO(response.content)
    ) as archive:
        archive.extractall(".")

    if os.path.exists("MagFace-main"):
        os.rename(
            "MagFace-main",
            MAGFACE_REPOSITORY_DIR,
        )

    if not os.path.exists(
        MAGFACE_REPOSITORY_DIR
    ):
        raise FileNotFoundError(
            "MagFace repository was not extracted "
            "to the expected directory."
        )

    # Preserve the package initialization required
    # by the original Colab workflow.
    os.makedirs(
        "MagFace/models/magface",
        exist_ok=True,
    )

    with open(
        "MagFace/models/__init__.py",
        "a",
    ):
        pass

    with open(
        "MagFace/models/magface/__init__.py",
        "a+",
    ) as file:
        import_line = (
            "from .magface import builder\n"
        )

        file.seek(0)
        existing_content = file.read()

        if import_line not in existing_content:
            file.write(import_line)

    print("✅ MagFace repository prepared.")


download_magface_repository()

## Load the MagFace ResNet-100 Model

The original experiment uses a **512-dimensional MagFace embedding** from a ResNet-100 (`iresnet100`) backbone.

The model configuration and pretrained-weight loading logic below are preserved from the working notebook.

No training step is performed.

In [ ]:
magface_root = os.path.abspath(
    MAGFACE_REPOSITORY_DIR
)

if magface_root not in sys.path:
    sys.path.append(
        magface_root
    )

builder_path = os.path.join(
    magface_root,
    "models",
    "magface.py",
)

if not os.path.exists(builder_path):
    raise FileNotFoundError(
        f"MagFace builder file not found: "
        f"{builder_path}"
    )

spec = importlib.util.spec_from_file_location(
    "magface_module",
    builder_path,
)

magface_module = (
    importlib.util.module_from_spec(spec)
)

spec.loader.exec_module(
    magface_module
)

builder = magface_module.builder

print("✅ MagFace builder imported.")

In [ ]:
def load_magface_resnet100(
    weight_path=MAGFACE_WEIGHTS_PATH
):
    """
    Construct the MagFace ResNet-100 backbone and load
    the pretrained weights used by the experiment.
    """

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    if not os.path.exists(weight_path):
        print(
            "❌ MagFace weights not found at: "
            f"{weight_path}"
        )
        return None

    print(
        "Initializing MagFace ResNet-100 "
        f"on {device}..."
    )

    try:
        # Configuration preserved from the original notebook.
        args = argparse.Namespace(
            backbone="iresnet100",
            arch="iresnet100",
            embedding_size=512,
            last_fc_size=512,
            arc_scale=64.0,
            l_margin=0.45,
            u_margin=0.8,
            l_a=10.0,
            u_a=110.0,
            resume=weight_path,
        )

        magface_model = builder(args)

        state_dict = torch.load(
            weight_path,
            map_location=device,
        )

        if "state_dict" in state_dict:
            state_dict = state_dict[
                "state_dict"
            ]

        cleaned_state_dict = {}

        for key, value in state_dict.items():
            name = key.replace(
                "module.",
                "",
            )

            if name.startswith("fc."):
                continue

            cleaned_state_dict[
                name
            ] = value

        magface_model.load_state_dict(
            cleaned_state_dict,
            strict=False,
        )

        magface_model = (
            magface_model
            .to(device)
            .eval()
        )

        # Preserve the helper attached by the original workflow.
        def get_feat_patch(img):
            img = img.astype(
                np.float32
            )

            img = (
                img - 127.5
            ) / 128.0

            img = np.transpose(
                img,
                (2, 0, 1),
            )

            img = (
                torch
                .from_numpy(img)
                .unsqueeze(0)
                .to(device)
            )

            with torch.no_grad():
                embedding = (
                    magface_model(img)
                )

                if isinstance(
                    embedding,
                    tuple,
                ):
                    embedding = embedding[0]

            return (
                embedding
                .cpu()
                .numpy()
            )

        magface_model.get_feat = (
            get_feat_patch
        )

        print(
            "✅ MagFace ResNet-100 ready "
            "for embedding extraction."
        )

        return magface_model

    except Exception as error:
        print(
            "❌ Error loading MagFace: "
            f"{error}"
        )
        return None

## Load the Pretrained Face-Recognition Models

The experiment evaluates three model conditions.

### ArcFace condition

The working notebook loads InsightFace's `antelopev2` model pack and uses its recognition component.

### MobileFaceNet condition

The working notebook loads InsightFace's `buffalo_sc` model pack and uses its recognition component.

### MagFace condition

The MagFace ResNet-100 backbone is loaded from the pretrained weights prepared above.

The terminology here deliberately uses **load** and **extract embeddings**, not **train**, because no new recognition model is trained in this project.

In [ ]:
INSIGHTFACE_MODEL_ROOT = (
    "/content/insightface_model"
)


def load_face_recognition_model(
    model_name,
    providers=None,
):
    """
    Load an InsightFace model pack and return its
    face-recognition component.
    """

    if providers is None:
        providers = [
            "CPUExecutionProvider"
        ]

    print(
        f"Initializing {model_name}..."
    )

    specific_model_path = os.path.join(
        INSIGHTFACE_MODEL_ROOT,
        "models",
        model_name,
    )

    nested_path = os.path.join(
        specific_model_path,
        model_name,
    )

    # Preserve the original compatibility fix for
    # model packs extracted into a nested folder.
    if os.path.exists(nested_path):
        print(
            f"Flattening nested {model_name} "
            "model directory..."
        )

        for item in os.listdir(
            nested_path
        ):
            shutil.move(
                os.path.join(
                    nested_path,
                    item,
                ),
                os.path.join(
                    specific_model_path,
                    item,
                ),
            )

        os.rmdir(
            nested_path
        )

    try:
        app_obj = FaceAnalysis(
            name=model_name,
            root=INSIGHTFACE_MODEL_ROOT,
            providers=providers,
        )

        app_obj.prepare(
            ctx_id=0,
            det_size=(640, 640),
        )

        recognition_model = None

        for key in [
            "recognition",
            "rec",
        ]:
            if key in app_obj.models:
                recognition_model = (
                    app_obj.models[key]
                )
                break

        if recognition_model is None:
            for model in (
                app_obj.models.values()
            ):
                if hasattr(
                    model,
                    "get_feat",
                ):
                    recognition_model = model
                    break

        if recognition_model is None:
            print(
                "⚠️ No recognition component "
                f"found for {model_name}."
            )
            return None, None

        print(
            f"✅ {model_name} loaded."
        )

        return (
            app_obj,
            recognition_model,
        )

    except Exception as error:
        print(
            f"⚠️ Could not load "
            f"{model_name}: {error}"
        )
        return None, None

In [ ]:
models_dict = {}

# ArcFace condition used by the original experiment.
app_arc, rec_arc = (
    load_face_recognition_model(
        "antelopev2"
    )
)

if app_arc is not None:
    models_dict["arcface"] = {
        "app": app_arc,
        "rec": rec_arc,
    }

# MobileFaceNet condition used by the original experiment.
app_mfn, rec_mfn = (
    load_face_recognition_model(
        "buffalo_sc"
    )
)

if app_mfn is not None:
    models_dict["mobilefacenet"] = {
        "app": app_mfn,
        "rec": rec_mfn,
    }

# MagFace pretrained model.
magface_backbone = (
    load_magface_resnet100(
        MAGFACE_WEIGHTS_PATH
    )
)

if magface_backbone is not None:
    models_dict["magface"] = {
        "app": None,
        "rec": magface_backbone,
    }

print("\nSuccessfully loaded model conditions:")

for model_name in models_dict:
    print(f" - {model_name}")

if not models_dict:
    raise RuntimeError(
        "No face-recognition models "
        "were loaded successfully."
    )

## Embedding Extraction Helper

`get_embedding_direct()` standardizes image decoding before passing each image to the appropriate pretrained recognition model.

### Image-format handling

- PNG/JPEG images are read directly with OpenCV.
- JPEG XL images are decoded to a temporary PNG using `djxl`.
- HEIC images are decoded to a temporary PNG using `heif-convert`.

### Input size

Images are resized to **112 × 112 pixels** when required.

### Model-specific extraction

- ArcFace / MobileFaceNet conditions use the InsightFace recognition model's `get_feat()` method.
- MagFace uses the PyTorch feature-extraction path from the original implementation.

The returned embedding vector is flattened before downstream cosine-similarity calculation.

In [ ]:
def get_embedding_direct(
    img_path,
    model,
    model_name,
):
    """
    Decode an image when required and extract its
    face embedding using a pretrained recognition model.
    """

    img_to_read = img_path

    # JPEG XL decoding.
    if img_path.lower().endswith(
        ".jxl"
    ):
        temp_png = (
            "/content/temp_decode.png"
        )

        if os.path.exists(DJXL_PATH):
            try:
                subprocess.run(
                    [
                        DJXL_PATH,
                        img_path,
                        temp_png,
                    ],
                    check=True,
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL,
                )

                img_to_read = temp_png

            except Exception:
                return None

    # HEIC decoding.
    elif img_path.lower().endswith(
        ".heic"
    ):
        temp_png = (
            "/content/temp_decode_heic.png"
        )

        try:
            subprocess.run(
                [
                    "heif-convert",
                    img_path,
                    temp_png,
                ],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )

            img_to_read = temp_png

        except Exception as error:
            print(
                "⚠️ HEIC decode failed for "
                f"{img_path}: {error}"
            )
            return None

    image = cv2.imread(
        img_to_read
    )

    if image is None:
        return None

    image_rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB,
    )

    if (
        image_rgb.shape[0] != 112
        or image_rgb.shape[1] != 112
    ):
        image_rgb = cv2.resize(
            image_rgb,
            (112, 112),
        )

    try:
        if model_name == "magface":
            # PyTorch MagFace extraction path
            # preserved from the original notebook.
            device = torch.device(
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

            transform = transforms.Compose(
                [
                    transforms.ToTensor(),
                    transforms.Normalize(
                        mean=[
                            0.5,
                            0.5,
                            0.5,
                        ],
                        std=[
                            0.5,
                            0.5,
                            0.5,
                        ],
                    ),
                ]
            )

            image_tensor = (
                transform(image_rgb)
                .unsqueeze(0)
                .to(device)
            )

            with torch.no_grad():
                output = model.features(
                    image_tensor
                )

                if isinstance(
                    output,
                    tuple,
                ):
                    embedding = output[0]
                else:
                    embedding = output

            return (
                embedding
                .cpu()
                .numpy()
                .flatten()
            )

        # InsightFace ONNX extraction path used
        # by ArcFace and MobileFaceNet conditions.
        embedding = model.get_feat(
            image_rgb
        )

        if (
            embedding is None
            or len(embedding) == 0
        ):
            return None

        return embedding.flatten()

    except Exception as error:
        print(
            "⚠️ Embedding extraction failed "
            f"for {model_name}: {error}"
        )
        return None

## Multi-Model Embedding Extraction and Pairwise Score Generation

This section preserves the original working evaluation flow.

For each pretrained model:

1. Extract embeddings from all L0 baseline images.
2. Generate the L0 genuine and impostor comparisons.
3. Extract embeddings from each compressed probe image.
4. Compare compressed probes only against baseline images from the **same demographic group**.
5. Calculate cosine similarity.
6. Label each pair as `genuine` or `impostor`.
7. Save all pairwise scores to `raw_pairwise_scores.csv`.

### Intra-group comparison rule

The original implementation intentionally restricts pairwise comparison to probe/baseline images from the same demographic group.

That matching rule is part of the experimental methodology and is preserved unchanged.

### L0 condition

For the uncompressed baseline:

- genuine = an L0 embedding compared with itself;
- impostor = an L0 embedding compared with other L0 embeddings from the same demographic group.

### Output columns

The generated raw score table contains:

```text
Model
Codec
Level
Race
Probe_ID
Baseline_ID
MatchType
Score
```

`Probe_ID` and `Baseline_ID` should still be audited before public release to ensure that the identifiers cannot be traced back to participant identities.

In [ ]:
formats = [
    "jpeg",
    "jxl",
    "heic",
]

levels = [
    "L1",
    "L2",
    "L3",
    "L4",
    "L5",
]

raw_scores_records = []

print(
    "Starting embedding extraction and "
    "strict intra-group pairwise scoring for:"
)

print(
    list(models_dict.keys())
)

for model_name, model_objs in (
    models_dict.items()
):
    print(
        f"\nProcessing model: "
        f"{model_name}"
    )

    current_model = (
        model_objs["rec"]
    )

    # -------------------------------------------------
    # Extract L0 baseline embeddings.
    # -------------------------------------------------
    current_baseline_embeddings = {}

    for root, _, files in os.walk(
        BASELINE_DIR
    ):
        for filename in files:
            if not filename.lower().endswith(
                (
                    ".png",
                    ".jpg",
                    ".jpeg",
                )
            ):
                continue

            file_id = os.path.splitext(
                filename
            )[0]

            baseline_group = (
                os.path.basename(root)
            )

            original_image_path = (
                os.path.join(
                    root,
                    filename,
                )
            )

            embedding = (
                get_embedding_direct(
                    original_image_path,
                    current_model,
                    model_name,
                )
            )

            if embedding is not None:
                current_baseline_embeddings[
                    file_id
                ] = {
                    "emb": embedding,
                    "race": baseline_group,
                }

    print(
        "   Loaded "
        f"{len(current_baseline_embeddings)} "
        "baseline embeddings."
    )

    # -------------------------------------------------
    # L0 uncompressed baseline scoring.
    # -------------------------------------------------
    print(
        "   Calculating L0 baseline "
        f"scores for {model_name}..."
    )

    uncompressed_codec = "NONE"
    uncompressed_level = "L0"

    baseline_items = list(
        current_baseline_embeddings.items()
    )

    for i in range(
        len(baseline_items)
    ):
        baseline_1_id, baseline_1_data = (
            baseline_items[i]
        )

        baseline_1_embedding = (
            baseline_1_data["emb"]
        )

        baseline_1_race = (
            baseline_1_data["race"]
        )

        # Genuine L0 score:
        # baseline embedding compared with itself.
        genuine_similarity = (
            cosine_similarity(
                baseline_1_embedding.reshape(
                    1,
                    -1,
                ),
                baseline_1_embedding.reshape(
                    1,
                    -1,
                ),
            )[0][0]
        )

        raw_scores_records.append(
            {
                "Model": model_name,
                "Codec": uncompressed_codec,
                "Level": uncompressed_level,
                "Race": (
                    baseline_1_race
                    .capitalize()
                ),
                "Probe_ID": baseline_1_id,
                "Baseline_ID": baseline_1_id,
                "MatchType": "genuine",
                "Score": genuine_similarity,
            }
        )

        # Impostor L0 scores:
        # compare with other baseline embeddings from
        # the same demographic group.
        for j in range(
            len(baseline_items)
        ):
            if i == j:
                continue

            (
                baseline_2_id,
                baseline_2_data,
            ) = baseline_items[j]

            baseline_2_embedding = (
                baseline_2_data["emb"]
            )

            baseline_2_race = (
                baseline_2_data["race"]
            )

            if (
                baseline_1_race
                == baseline_2_race
            ):
                impostor_similarity = (
                    cosine_similarity(
                        baseline_1_embedding.reshape(
                            1,
                            -1,
                        ),
                        baseline_2_embedding.reshape(
                            1,
                            -1,
                        ),
                    )[0][0]
                )

                raw_scores_records.append(
                    {
                        "Model": model_name,
                        "Codec": (
                            uncompressed_codec
                        ),
                        "Level": (
                            uncompressed_level
                        ),
                        "Race": (
                            baseline_1_race
                            .capitalize()
                        ),
                        "Probe_ID": (
                            baseline_1_id
                        ),
                        "Baseline_ID": (
                            baseline_2_id
                        ),
                        "MatchType": (
                            "impostor"
                        ),
                        "Score": (
                            impostor_similarity
                        ),
                    }
                )

    print(
        "   L0 baseline scores "
        f"calculated for {model_name}."
    )

    # -------------------------------------------------
    # Compressed probe extraction and scoring.
    # -------------------------------------------------
    for image_format in formats:
        for level in levels:
            level_path = os.path.join(
                PROBE_DIR,
                image_format,
                level,
            )

            if not os.path.exists(
                level_path
            ):
                continue

            for root, _, files in os.walk(
                level_path
            ):
                image_files = [
                    filename
                    for filename in files
                    if filename.lower().endswith(
                        (
                            ".png",
                            ".jpg",
                            ".jpeg",
                            ".jxl",
                            ".heic",
                        )
                    )
                ]

                for filename in image_files:
                    probe_group = (
                        os.path.basename(root)
                    )

                    # Preserve the original probe-ID
                    # extraction logic.
                    parts = (
                        os.path.splitext(filename)[0]
                        .split("_")
                    )

                    if len(parts) >= 3:
                        probe_id = (
                            f"{parts[1]}_"
                            f"{parts[2]}"
                        )

                    elif len(parts) >= 2:
                        probe_id = parts[0]

                    else:
                        probe_id = (
                            os.path.splitext(
                                filename
                            )[0]
                        )

                    probe_image_path = (
                        os.path.join(
                            root,
                            filename,
                        )
                    )

                    probe_embedding = (
                        get_embedding_direct(
                            probe_image_path,
                            current_model,
                            model_name,
                        )
                    )

                    if probe_embedding is None:
                        continue

                    for (
                        baseline_id,
                        baseline_data,
                    ) in (
                        current_baseline_embeddings
                        .items()
                    ):
                        baseline_embedding = (
                            baseline_data["emb"]
                        )

                        baseline_race = (
                            baseline_data["race"]
                        )

                        # Preserve the original strict
                        # intra-group comparison rule.
                        if (
                            probe_group
                            != baseline_race
                        ):
                            continue

                        similarity = (
                            cosine_similarity(
                                probe_embedding.reshape(
                                    1,
                                    -1,
                                ),
                                baseline_embedding.reshape(
                                    1,
                                    -1,
                                ),
                            )[0][0]
                        )

                        match_type = (
                            "genuine"
                            if probe_id
                            == baseline_id
                            else "impostor"
                        )

                        raw_scores_records.append(
                            {
                                "Model": model_name,
                                "Codec": (
                                    image_format
                                    .upper()
                                ),
                                "Level": level,
                                "Race": (
                                    probe_group
                                    .capitalize()
                                ),
                                "Probe_ID": (
                                    probe_id
                                ),
                                "Baseline_ID": (
                                    baseline_id
                                ),
                                "MatchType": (
                                    match_type
                                ),
                                "Score": (
                                    similarity
                                ),
                            }
                        )

df_raw = pd.DataFrame(
    raw_scores_records
)

df_raw.to_csv(
    RAW_SCORES_PATH,
    index=False,
)

print(
    "\n✅ Embedding extraction and "
    "pairwise score generation complete."
)

print(
    f"Saved raw scores to: "
    f"{RAW_SCORES_PATH}"
)

print(
    f"Total pairwise records: "
    f"{len(df_raw):,}"
)

## Output of This Stage

The main output of this notebook is:

```text
results/
└── data/
    └── raw_pairwise_scores.csv
```

This file contains cosine-similarity comparisons generated from embeddings extracted with the three pretrained face-recognition model conditions.

The next notebook, **`04_score_generation_and_fairness.ipynb`**, uses these raw scores to calculate recognition-performance and demographic-fairness metrics.
